# RF-DETR Inference Latency Benchmark

Measures inference latency for three RF-DETR families across three configs:

| Config | Description |
|--------|-------------|
| **FP32** | `predict()` — unoptimized baseline |
| **FP16+JIT** | `optimize_for_inference(dtype=torch.float16)` |
| **ONNX** | exported `.onnx` via `onnxruntime-gpu` |

## 1. Install

In [ ]:
!pip install -q "rfdetr[onnx]" onnxruntime-gpu pillow

## 2. Config

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import Any, NamedTuple

import numpy as np
import torch
from PIL import Image

WARMUP_RUNS = 20
MEASURE_RUNS = 100
EXPORT_DIR = Path("benchmark_output")
EXPORT_DIR.mkdir(exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("This benchmark requires a CUDA GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Sample images

In [ ]:
rng = np.random.default_rng(42)
images: list[Image.Image] = [Image.fromarray(rng.integers(0, 256, (640, 640, 3), dtype=np.uint8)) for _ in range(10)]
print(f"Generated {len(images)} synthetic 640×640 RGB images")

## 4. Latency helpers

In [ ]:
class BenchmarkResult(NamedTuple):
    """Single benchmark measurement."""

    label: str
    mean_ms: float
    std_ms: float

    @property
    def fps(self) -> float:
        """Frames per second."""
        return 1000.0 / self.mean_ms


def measure_latency_gpu(
    fn: Callable[[], object],
    warmup: int = WARMUP_RUNS,
    runs: int = MEASURE_RUNS,
) -> tuple[float, float]:
    """Return (mean_ms, std_ms) using CUDA events."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    timings: list[float] = []
    for _ in range(runs):
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        timings.append(start.elapsed_time(end))
    arr = np.array(timings)
    return float(arr.mean()), float(arr.std())


_measure = measure_latency_gpu

## 5. Per-config benchmark functions

In [ ]:
def _predict_fp32(model: Any, image: Image.Image) -> BenchmarkResult:
    """Baseline FP32 predict() latency."""
    mean, std = _measure(lambda: model.predict(image))
    return BenchmarkResult("predict() FP32", mean, std)


def _predict_fp16(model: Any, image: Image.Image) -> BenchmarkResult:
    """FP16+JIT latency — applies and removes optimize_for_inference."""
    model.optimize_for_inference(dtype=torch.float16)
    mean, std = _measure(lambda: model.predict(image))
    model.remove_optimized_model()
    return BenchmarkResult("predict() FP16+JIT", mean, std)


def _onnx_rt(model: Any, image: Image.Image, export_dir: Path) -> BenchmarkResult:
    """Export to ONNX and return ORT inference latency."""
    from rfdetr.export._onnx.inference import _create_onnx_session

    onnx_path = model.export(output_dir=str(export_dir))
    sess = _create_onnx_session(onnx_path)

    input_meta = sess.get_inputs()[0]
    _, _channels, height, width = input_meta.shape
    _mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    _std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    arr = np.array(image.resize((width, height), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0
    inp = ((arr.transpose(2, 0, 1) - _mean) / _std)[np.newaxis]

    mean_ms, std_ms = _measure(lambda: sess.run(None, {input_meta.name: inp}))
    provider_label = sess.get_providers()[0].replace("ExecutionProvider", "")
    return BenchmarkResult(f"ONNX ({provider_label})", mean_ms, std_ms)

## 6. Model benchmark runner

In [ ]:
def run_model_benchmark(
    model_cls: type,
    model_name: str,
    images: list[Image.Image],
) -> list[BenchmarkResult]:
    """Run FP32 / FP16 / ONNX benchmarks for one model and print results."""
    print(f"\n{'=' * 62}")
    print(f"  {model_name}")
    print("=" * 62)

    model: Any = model_cls()
    export_dir = EXPORT_DIR / model_name.split()[0]
    export_dir.mkdir(exist_ok=True)
    image = images[0]

    fp32 = _predict_fp32(model, image)
    fp16 = _predict_fp16(model, image)
    results: list[BenchmarkResult] = [fp32, fp16]

    try:
        results.append(_onnx_rt(model, image, export_dir))
    except Exception as exc:
        print(f"  ⚠ ONNX skipped: {exc}")

    for r in results:
        print(f"  {r.label:<30}  {r.mean_ms:6.2f} ms ± {r.std_ms:5.2f}   ({r.fps:6.1f} FPS)")

    onnx_results = [r for r in results if r.label.startswith("ONNX")]
    speedups = [f"FP16 {fp32.mean_ms / fp16.mean_ms:.1f}×"]
    if onnx_results:
        speedups.append(f"ONNX {fp32.mean_ms / onnx_results[0].mean_ms:.1f}×")
    print(f"  Speedup vs FP32: {' | '.join(speedups)}")

    return results

## 7. Benchmark loop — detection · segmentation · keypoint

In [ ]:
from rfdetr import RFDETRKeypointPreview, RFDETRMedium, RFDETRSegSmall

MODELS: list[tuple[type, str]] = [
    (RFDETRMedium, "RFDETRMedium — detection"),
    (RFDETRSegSmall, "RFDETRSegSmall — segmentation"),
    (RFDETRKeypointPreview, "RFDETRKeypointPreview — keypoint"),
]

all_results: dict[str, list[BenchmarkResult]] = {}
for _model_cls, _model_name in MODELS:
    all_results[_model_name] = run_model_benchmark(_model_cls, _model_name, images)

## 8. Summary

In [ ]:
print("\n" + "=" * 70)
print(f"  {'Model':<36}  {'Config':<22}  {'FPS':>7}")
print("=" * 70)
for _model_name, _results in all_results.items():
    for _r in _results:
        print(f"  {_model_name:<36}  {_r.label:<22}  {_r.fps:7.1f}")
print("=" * 70)
print(f"\nRuns: {MEASURE_RUNS} timed + {WARMUP_RUNS} warmup.  Batch 1.  GPU: {torch.cuda.get_device_name(0)}.")